# REINFORCE で CartPole を解く(演習)

解説資料は `research-handbook/reinforcement-learning/policy-gradient.md`。
このnotebookは**直接編集せず**、自分の卒研repoにコピーして使うこと(手順は `technical-handbook/colab/use-github-repo.md`)。

「---- ここから課題 ----」の区間を埋めながら上から順に実行する。
解答付き版は `research-handbook/notebooks/reinforce_cartpole_solution.ipynb` にあるが、まず自力で取り組むこと。

方策を直接パラメータ化し、方策勾配

$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\left[ \sum_t G_t \, \nabla_\theta \log \pi_\theta(a_t \mid s_t) \right]$$

に従ってパラメータを更新する。ニューラルネットを使わず**線形softmax方策 + numpy** で実装するので、勾配の中身を手で確認できる。

In [ ]:
# Colab の場合は初回のみ実行してください
# !pip install -q gymnasium
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

env = gym.make("CartPole-v1")
obs_dim = 4      # [カート位置, カート速度, ポール角度, ポール角速度]
n_actions = 2    # 0=左に押す, 1=右に押す
print("observation:", obs_dim, "actions:", n_actions)

## 方策の定義: 線形softmax方策

$$\pi_W(a \mid s) = \frac{\exp(W_a^\top s)}{\sum_{a'} \exp(W_{a'}^\top s)}$$

バンディットで学んだBoltzmann方策(`bandit.md` 5.3節)の「$Q$ 値」を「状態の線形変換」に置き換えた形です。

In [ ]:
def softmax(x):
    x = x - np.max(x)            # オーバーフロー対策
    e = np.exp(x)
    return e / e.sum()

def policy(W, s):
    """線形softmax方策 pi(a|s) = softmax(W s)。W: (n_actions, obs_dim)"""
    return softmax(W @ s)

def sample_action(W, s, rng):
    p = policy(W, s)
    return rng.choice(n_actions, p=p), p

## 課題1: $\nabla_W \log \pi(a \mid s)$ の実装

softmax方策の対数勾配は次の形になります(導出は `policy-gradient.md` 参照):

$$\nabla_{W_k} \log \pi(a \mid s) = (\mathbb{1}[a = k] - \pi(k \mid s))\, s$$

実装したら**数値微分と比較**して正しさを確認します。勾配実装のバグは学習が「なんとなく遅い」形で現れるため、この検算の習慣が重要です。

In [ ]:
def grad_log_pi(W, s, a):
    """課題1: ∇_W log pi(a|s) を (n_actions, obs_dim) の行列で返す"""
    # ---- ここから課題1 ----
    # policy-gradient.md の導出を参照。np.outer が便利
    # 完成したら下の数値微分チェックで検算すること


    # ---- ここまで ----

# 勾配の数値チェック(有限差分と比較)
rng = np.random.default_rng(0)
W = rng.normal(size=(n_actions, obs_dim)) * 0.1
s = rng.normal(size=obs_dim)
a = 1
g = grad_log_pi(W, s, a)
eps = 1e-6
num = np.zeros_like(W)
for i in range(n_actions):
    for j in range(obs_dim):
        Wp = W.copy(); Wp[i, j] += eps
        Wm = W.copy(); Wm[i, j] -= eps
        num[i, j] = (np.log(policy(Wp, s)[a]) - np.log(policy(Wm, s)[a])) / (2 * eps)
print("解析勾配と数値勾配の最大誤差:", np.abs(g - num).max())

## 課題2: 収益 $G_t$ の計算

$G_t = r_{t+1} + \gamma G_{t+1}$ を利用してエピソードの**後ろから**計算します(前から計算すると $O(T^2)$ になる)。

In [ ]:
def compute_returns(rewards, gamma=0.99):
    """課題2: 各時刻の収益 G_t を返す"""
    G = np.zeros(len(rewards))
    # ---- ここから課題2 ----
    # G_t = r_{t+1} + gamma * G_{t+1} の関係を使い、エピソードの後ろから計算する
    # (前から素朴に計算すると O(T^2) になってしまう)


    # ---- ここまで ----
    return G

print(compute_returns([1.0, 1.0, 1.0], gamma=0.9))  # [2.71, 1.9, 1.0] になるはず

## 課題3: REINFORCE本体

エピソードを1本生成 → 各時刻の $G_t$ を計算 → $W \leftarrow W + \alpha G_t \nabla_W \log \pi(a_t \mid s_t)$。
収益の標準化(平均を引いて標準偏差で割る)は**ベースライン + スケール調整**にあたり、学習を大きく安定させます。

In [ ]:
def run_reinforce(n_episodes=800, alpha=0.01, gamma=0.99, seed=0, use_baseline=True):
    """課題3: REINFORCE(ベースライン付き)"""
    rng = np.random.default_rng(seed)
    W = rng.normal(size=(n_actions, obs_dim)) * 0.01
    history = []
    for ep in range(n_episodes):
        states, actions, rewards = [], [], []
        s, _ = env.reset(seed=int(rng.integers(1_000_000)))
        done = False
        while not done:
            a, _ = sample_action(W, s, rng)
            s_next, r, terminated, truncated, _ = env.step(a)
            states.append(s); actions.append(a); rewards.append(r)
            s = s_next
            done = terminated or truncated
        G = compute_returns(rewards, gamma)
        if use_baseline:
            # 定数ベースライン(平均収益)を引いて分散を下げる。標準化はさらに安定
            G = (G - G.mean()) / (G.std() + 1e-8)
        for s_t, a_t, G_t in zip(states, actions, G):
            # ---- ここから課題3 ----
            # 方策勾配の上昇: W を alpha * G_t * ∇_W log pi(a_t|s_t) だけ動かす
            pass
            # ---- ここまで ----
        history.append(np.sum(rewards))
    return W, np.array(history)

W_learned, hist = run_reinforce()
print(f"最初の50ep平均: {hist[:50].mean():.1f} → 最後の50ep平均: {hist[-50:].mean():.1f}")

In [ ]:
def moving_average(x, w=20):
    return np.convolve(x, np.ones(w) / w, mode="valid")

plt.figure(figsize=(8, 4))
plt.plot(hist, alpha=0.3, label="episode return")
plt.plot(moving_average(hist), label="moving average (20ep)")
plt.xlabel("episode"); plt.ylabel("return")
plt.legend(); plt.title("REINFORCE on CartPole-v1")
plt.show()

## 発展課題

1. ベースラインあり/なしを**複数seedの平均**で比較し、分散低減の効果を確認する(下のセル)
2. $\alpha$ を10倍/10分の1にすると学習曲線はどうなるか(`hyperparameter.md` の症状表と対応づける)
3. 割引率 $\gamma$ を 1.0 にすると何が起きるか考察する

In [ ]:
# 発展課題: ベースラインなしとの比較(複数seedで平均すること)
returns_bl, returns_nobl = [], []
for seed in range(3):
    _, h1 = run_reinforce(n_episodes=300, seed=seed, use_baseline=True)
    _, h0 = run_reinforce(n_episodes=300, seed=seed, use_baseline=False)
    returns_bl.append(h1); returns_nobl.append(h0)
plt.figure(figsize=(8, 4))
plt.plot(moving_average(np.mean(returns_bl, axis=0)), label="with baseline")
plt.plot(moving_average(np.mean(returns_nobl, axis=0)), label="without baseline")
plt.xlabel("episode"); plt.ylabel("return (3-seed mean)")
plt.legend(); plt.title("Effect of baseline")
plt.show()

## まとめ

- REINFORCEは「たくさん報酬をもらえた行動の対数確率を上げる」だけの素直なアルゴリズム
- ただし分散が大きく、ベースラインによる分散低減がほぼ必須
- ベースラインを学習する価値関数に置き換えると **actor-critic**(`policy-gradient.md` 後半)になる
- ニューラルネット化・大規模化した現代版が **PPO**(`ppo-mppi.md`)